# Lab 1: Deploy Medical MCP Server Lambda

이 노트북에서는 `/mcp` 디렉토리의 MCP 서버 코드를 Lambda 함수로 패키징하고 배포합니다.

## Architecture
```
AgentCore Gateway → Lambda (MCP Tool Handler) → EMR Serverless (Livy) → S3 Tables
```

## Step 1: Configuration

아래 변수를 환경에 맞게 수정하세요.

In [ ]:
import boto3
import json
import time
import os
import zipfile
import io

REGION = "us-west-2"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
FUNCTION_NAME = "fhir-mcp-server"
LAMBDA_ROLE_NAME = "fhir-mcp-lambda-role"

# EMR Serverless - CDK에서 배포된 리소스에서 가져옵니다
emr_client = boto3.client("emr-serverless", region_name=REGION)
apps = emr_client.list_applications()["applications"]
emr_app = next(a for a in apps if a["name"] == "fhir-data-query")
EMR_APPLICATION_ID = emr_app["id"]

# EMR Execution Role ARN - CDK에서 생성된 역할
iam_client = boto3.client("iam")
roles = iam_client.list_roles(MaxItems=200)["Roles"]
emr_exec_role = next(r for r in roles if "EmrServerlessExecution" in r["RoleName"])
EMR_EXECUTION_ROLE_ARN = emr_exec_role["Arn"]

print(f"Account: {ACCOUNT_ID}")
print(f"EMR Application ID: {EMR_APPLICATION_ID}")
print(f"EMR Execution Role: {EMR_EXECUTION_ROLE_ARN}")

## Step 2: Create Lambda Execution Role

Lambda가 EMR Serverless Livy endpoint에 접근하고 CloudWatch Logs에 기록할 수 있는 IAM 역할을 생성합니다.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["emr-serverless:AccessLivyEndpoints", "emr-serverless:GetApplication"],
            "Resource": f"arn:aws:emr-serverless:{REGION}:{ACCOUNT_ID}:/applications/{EMR_APPLICATION_ID}"
        },
        {
            "Effect": "Allow",
            "Action": "iam:PassRole",
            "Resource": EMR_EXECUTION_ROLE_ARN,
            "Condition": {"StringLike": {"iam:PassedToService": "emr-serverless.amazonaws.com"}}
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*"
        }
    ]
}

try:
    role = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy)
    )
    print(f"Created role: {LAMBDA_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role already exists: {LAMBDA_ROLE_NAME}")

iam_client.put_role_policy(
    RoleName=LAMBDA_ROLE_NAME,
    PolicyName="EMRServerlessLivyAccess",
    PolicyDocument=json.dumps(inline_policy)
)

LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"
print(f"Role ARN: {LAMBDA_ROLE_ARN}")
print("Waiting 10s for IAM propagation...")
time.sleep(10)

## Step 3: Package Lambda Function

`/mcp` 디렉토리의 코드를 ZIP으로 패키징합니다. `botocore`와 `awscrt`는 Lambda 런타임에 포함되어 있고, `requests`만 번들링합니다.

In [ ]:
import subprocess
import shutil

MCP_DIR = "../mcp"  # 노트북 기준 상대 경로
BUILD_DIR = "/tmp/mcp-lambda-build"
ZIP_PATH = "/tmp/mcp-lambda.zip"

# Clean build directory
if os.path.exists(BUILD_DIR):
    shutil.rmtree(BUILD_DIR)
os.makedirs(BUILD_DIR)

# Install requests into build dir
subprocess.run(["pip", "install", "requests", "-t", BUILD_DIR, "--quiet"], check=True)

# Copy MCP source code
for item in ["handler.py", "emr_client.py"]:
    shutil.copy2(os.path.join(MCP_DIR, item), BUILD_DIR)
shutil.copytree(os.path.join(MCP_DIR, "tools"), os.path.join(BUILD_DIR, "tools"), dirs_exist_ok=True)

# Create ZIP
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(BUILD_DIR):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, os.path.relpath(full, BUILD_DIR))

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Lambda package: {ZIP_PATH} ({size_mb:.1f} MB)")

## Step 4: Deploy Lambda Function

Lambda 함수를 생성하거나 업데이트합니다. EMR Serverless Livy 세션은 응답까지 시간이 걸릴 수 있으므로 타임아웃을 5분으로 설정합니다.

In [ ]:
lambda_client = boto3.client("lambda", region_name=REGION)

with open(ZIP_PATH, "rb") as f:
    zip_bytes = f.read()

env_vars = {
    "EMR_APPLICATION_ID": EMR_APPLICATION_ID,
    "EMR_EXECUTION_ROLE_ARN": EMR_EXECUTION_ROLE_ARN,
    "AWS_REGION_NAME": REGION,
}

try:
    resp = lambda_client.create_function(
        FunctionName=FUNCTION_NAME,
        Runtime="python3.13",
        Role=LAMBDA_ROLE_ARN,
        Handler="handler.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=300,
        MemorySize=512,
        Environment={"Variables": env_vars},
    )
    print(f"Created Lambda: {resp['FunctionArn']}")
except lambda_client.exceptions.ResourceConflictException:
    resp = lambda_client.update_function_code(
        FunctionName=FUNCTION_NAME, ZipFile=zip_bytes
    )
    lambda_client.update_function_configuration(
        FunctionName=FUNCTION_NAME,
        Timeout=300,
        MemorySize=512,
        Environment={"Variables": env_vars},
    )
    print(f"Updated Lambda: {resp['FunctionArn']}")

LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{FUNCTION_NAME}"
print(f"\nLambda ARN: {LAMBDA_ARN}")

## Step 5: Test Lambda (Optional)

Lambda 함수가 정상 동작하는지 `list_tables` 도구로 테스트합니다.

> ⚠️ 첫 호출 시 Livy 세션 생성에 1-2분 소요될 수 있습니다.

In [ ]:
test_event = {
    "toolName": "list_tables",
    "arguments": {"domain": "clinical"}
}

print("Invoking Lambda (may take 1-2 min for first call)...")
resp = lambda_client.invoke(
    FunctionName=FUNCTION_NAME,
    Payload=json.dumps(test_event)
)
result = json.loads(resp["Payload"].read())
print(json.dumps(result, indent=2, ensure_ascii=False))

## ✅ Done!

Lambda 함수가 배포되었습니다. 다음 노트북 (`02_setup_agentcore_gateway.ipynb`)에서 AgentCore Gateway를 설정합니다.

### 배포된 리소스
- **Lambda Function**: `fhir-mcp-server`
- **IAM Role**: `fhir-mcp-lambda-role`
- **13 MCP Tools**: patient, clinical, financial, analytics, schema discovery, custom query